# Automated IEEE Introduction Writer for Indoor Testbed Paper
## Complete pipeline for analyzing papers and generating introduction with MLA citations

In [ ]:
# Import Required Libraries
import os
import glob
import time
import json
import re
from typing import List, Dict, Tuple
import getpass
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# API Keys
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")

# Initialize LLMs with different temperatures for different tasks
llm_analyzer = ChatOpenAI(model="gpt-4-turbo-preview", temperature=0.1)
llm_writer = ChatOpenAI(model="gpt-4-turbo-preview", temperature=0.2)

In [ ]:
# Full Paper Context
PAPER_CONTEXT = """
Title: Indoor Testbed for Multirobot Research with Microdrones

Key Contributions:
1. Extended SCU RSL testbed from ground rovers to 3D environments with Crazyflie microdrones
2. Achieved sub-centimeter OptiTrack tracking accuracy with 24-camera configuration
3. Demonstrated 1m formation spacing accuracy, 10cm altitude precision, 20-degree heading accuracy
4. Integrated cluster control architecture for multi-drone coordination at 100Hz update rates
5. Enabled single-operator multi-drone control by circumventing FAA regulations indoors
6. Validated system with hover, dynamic positioning, and formation shrinking maneuvers

System Specifications:
- Workspace: 2.5m x 6m x 1.3m (scalable to 3m x 6m x 6m)
- Drones: Crazyflie 2.1 (92x92x29mm, <250g with markers)
- Tracking: OptiTrack Flex 3 cameras, 640x480px, 100fps
- Control: ROS2 + ArduPilot integration
- Communication: 2.4GHz radio via Crazyradio PA

Previous SCU Work:
- Cluster control methodology (Kitts & Mas 2009)
- AR.Drone testbed at NASA Ames (Cashbaugh et al. 2015)
- Multi-platform implementations (land, sea, air)

This testbed bridges simulation-to-reality gap for:
- Vector field navigation research
- Adaptive navigation algorithms
- Environmental monitoring applications
- Formation control validation
"""

In [ ]:
def load_and_process_pdfs(folder_path="./PapersForResearch"):
    """Load PDFs with metadata extraction for citations"""
    pdf_paths = glob.glob(f"{folder_path}/*.pdf")
    papers = []
    
    for i, path in enumerate(pdf_paths, 1):
        print(f"Loading {i}/{len(pdf_paths)}: {os.path.basename(path)}")
        try:
            loader = PyPDFLoader(path)
            pages = loader.load()
            
            # Extract title and authors from first page
            first_page = pages[0].page_content if pages else ""
            
            # Get title (usually in first few lines)
            lines = first_page.split('\n')[:10]
            title = lines[0] if lines else os.path.basename(path).replace('.pdf', '')
            
            # Extract year from filename or content
            year_match = re.search(r'(19|20)\d{2}', path)
            year = year_match.group() if year_match else "2024"
            
            # Combine key pages (abstract, intro, conclusion)
            content = ""
            for page_num, page in enumerate(pages[:15]):  # First 15 pages
                content += f"\n--- Page {page_num + 1} ---\n" + page.page_content
            
            papers.append({
                "id": i,
                "filename": os.path.basename(path),
                "title": title.strip(),
                "year": year,
                "first_page": first_page[:2000],
                "content": content[:40000]  # Limit for token management
            })
            
        except Exception as e:
            print(f"Error loading {path}: {e}")
    
    return papers

In [ ]:
# First Analysis Pass: Extract relevant content and citations
analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are analyzing papers for an indoor microdrone testbed paper.
    Focus on: testbeds, cluster control, formation control, indoor drones, feature finding, 
    motion capture systems, indoor testing, FAA regulations, simulation-to-reality gap,
    multi-operator challenges, and multirobot coordination."""),
    ("human", """Analyze this paper and extract:
    
    1. AUTHORS: List all authors (Last, First format)
    2. TITLE: Full paper title
    3. VENUE: Conference/journal name if visible
    4. YEAR: Publication year
    
    5. KEY CONTRIBUTION: Main contribution in one sentence
    
    6. RELEVANT QUOTES (provide exact quotes with context):
       a) Problems/gaps in multirobot systems
       b) Indoor testbed benefits or challenges
       c) Cluster vs swarm control discussion
       d) Regulatory/safety considerations
       e) Formation control achievements
       f) Hardware-in-the-loop testing importance
    
    7. RELEVANCE SCORE (1-10): How relevant to indoor multirobot testbeds?
    
    Paper content:
    {paper_content}
    """)
])

def analyze_papers(papers):
    """Extract citations and relevant content"""
    analyzed = []
    
    for paper in papers:
        print(f"\nAnalyzing [{paper['id']}]: {paper['filename'][:50]}...")
        
        chain = analysis_prompt | llm_analyzer
        try:
            response = chain.invoke({"paper_content": paper['content']})
            
            paper['analysis'] = response.content
            paper['citation_num'] = paper['id']
            analyzed.append(paper)
            
            print(f"  ✓ Analyzed successfully")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
            paper['analysis'] = f"Error analyzing: {e}"
            analyzed.append(paper)
        
        time.sleep(0.5)  # Rate limiting
    
    return analyzed

In [ ]:
# Second Pass: Generate targeted questions based on gaps
question_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are identifying key themes and gaps for positioning a microdrone testbed paper.
    The paper combines indoor testbed infrastructure with cluster control methodology."""),
    ("human", """Based on these paper analyses, identify:
    
    1. What specific problems do researchers face moving from simulation to hardware?
    2. What regulatory/operational challenges exist for multi-drone testing?
    3. How does cluster control differ from swarm approaches in practice?
    4. What tracking accuracy is needed for formation control?
    5. Why is indoor testing valuable despite GPS limitations?
    6. What enables single-operator control of multiple vehicles?
    
    Paper summaries:
    {summaries}
    
    Generate 5-7 specific positioning statements our testbed addresses.
    """)
])

def generate_positioning(analyzed_papers):
    """Identify key positioning for our contribution"""
    # Get high-relevance papers
    relevant = [p for p in analyzed_papers if "RELEVANCE SCORE" in p.get('analysis', '')]
    summaries = "\n\n".join([f"[{p['citation_num']}]: {p['analysis'][:1000]}" 
                            for p in relevant[:15]])
    
    chain = question_prompt | llm_analyzer
    response = chain.invoke({"summaries": summaries})
    
    return response.content

In [ ]:
# Introduction Writer
intro_prompt = ChatPromptTemplate.from_messages([
    ("system", """Write an IEEE Transactions introduction for an indoor microdrone testbed paper.
    Style: Formal, technical, methodical progression from broad to specific.
    Structure: 
    1. Broad multirobot benefits and architectures
    2. Simulation vs hardware challenges
    3. Indoor testbed advantages
    4. Cluster control benefits
    5. Our specific contributions
    6. Paper organization
    Use [1], [2-4] style citations. No em-dashes. SCU RSL style."""),
    ("human", """Write a 2-page introduction incorporating:
    
    PAPER CONTEXT:
    {context}
    
    KEY POSITIONING:
    {positioning}
    
    ANALYZED PAPERS WITH QUOTES:
    {analyses}
    
    Requirements:
    - Start: "Mobile multirobot systems have become increasingly utilized..."
    - Emphasize both testbed infrastructure AND cluster control integration
    - Explain how indoor operation enables single-operator control
    - Include 30-40 citations from provided papers
    - End with paper organization paragraph
    - Maintain formal IEEE tone throughout
    """)
])

def write_introduction(analyzed_papers, positioning):
    """Generate the complete introduction"""
    # Format analyses with citations
    analyses = ""
    for p in analyzed_papers:
        if 'analysis' in p:
            analyses += f"\n[{p['citation_num']}] - {p['filename']}:\n"
            analyses += p['analysis'][:2000] + "\n"
            analyses += "-" * 50 + "\n"
    
    chain = intro_prompt | llm_writer
    response = chain.invoke({
        "context": PAPER_CONTEXT,
        "positioning": positioning,
        "analyses": analyses
    })
    
    return response.content

In [ ]:
def extract_mla_citations(analyzed_papers):
    """Generate MLA format citations"""
    citations = []
    
    for p in analyzed_papers:
        analysis = p.get('analysis', '')
        
        # Extract author info
        author_match = re.search(r'AUTHORS?:(.+?)(?:TITLE|$)', analysis, re.IGNORECASE | re.DOTALL)
        authors = author_match.group(1).strip() if author_match else "Unknown Author"
        
        # Extract title
        title_match = re.search(r'TITLE:(.+?)(?:VENUE|YEAR|$)', analysis, re.IGNORECASE | re.DOTALL)
        title = title_match.group(1).strip() if title_match else p.get('title', p['filename'])
        
        # Extract venue
        venue_match = re.search(r'VENUE:(.+?)(?:YEAR|$)', analysis, re.IGNORECASE | re.DOTALL)
        venue = venue_match.group(1).strip() if venue_match else "Conference Proceedings"
        
        # Clean up authors for MLA (Last, First format)
        authors_clean = authors.split(',')[0].strip() if ',' in authors else authors.split()[0] if authors != "Unknown Author" else authors
        
        # Format MLA citation
        if "et al" in authors or len(authors.split(',')) > 2:
            citation = f"[{p['citation_num']}] {authors_clean}, et al."
        else:
            citation = f"[{p['citation_num']}] {authors_clean}."
        
        citation += f' "{title}." {venue}, {p["year"]}.'
        
        citations.append(citation)
    
    return sorted(citations, key=lambda x: int(re.search(r'\[(\d+)\]', x).group(1)))

In [ ]:
# Main Pipeline Execution
def run_complete_pipeline():
    """Execute the full introduction writing pipeline"""
    
    print("="*60)
    print("INDOOR TESTBED PAPER - INTRODUCTION GENERATION PIPELINE")
    print("="*60)
    
    # Step 1: Load Papers
    print("\n[1/5] Loading PDF papers...")
    papers = load_and_process_pdfs()
    print(f"→ Loaded {len(papers)} papers successfully")
    
    # Step 2: Analyze Papers
    print("\n[2/5] Analyzing papers for relevant content...")
    analyzed = analyze_papers(papers)
    print(f"→ Completed analysis of {len(analyzed)} papers")
    
    # Step 3: Generate Positioning
    print("\n[3/5] Identifying key positioning statements...")
    positioning = generate_positioning(analyzed)
    print("→ Positioning statements generated:")
    print(positioning[:500] + "...")
    
    # Step 4: Write Introduction
    print("\n[4/5] Writing IEEE introduction...")
    introduction = write_introduction(analyzed, positioning)
    print("→ Introduction completed")
    
    # Step 5: Generate Citations
    print("\n[5/5] Generating MLA citations...")
    citations = extract_mla_citations(analyzed)
    print(f"→ Generated {len(citations)} citations")
    
    # Save outputs
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Save introduction
    intro_file = f"introduction_{timestamp}.txt"
    with open(intro_file, 'w', encoding='utf-8') as f:
        f.write("%% IEEE Introduction for Indoor Testbed Paper\n")
        f.write("%% Generated: " + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + "\n\n")
        f.write("\\section{Introduction}\n\n")
        f.write(introduction)
    
    # Save citations
    citations_file = f"references_{timestamp}.txt"
    with open(citations_file, 'w', encoding='utf-8') as f:
        f.write("%% References in MLA Format\n\n")
        f.write("\\begin{thebibliography}{99}\n\n")
        for citation in citations:
            # Format for LaTeX bibliography
            bib_entry = citation.replace('[', '\\bibitem{').replace(']', '}')
            f.write(bib_entry + "\n\n")
        f.write("\\end{thebibliography}\n")
    
    print("\n" + "="*60)
    print("PIPELINE COMPLETE")
    print("="*60)
    print(f"✓ Introduction saved to: {intro_file}")
    print(f"✓ References saved to: {citations_file}")
    print("\nFiles are ready for LaTeX compilation.")
    
    return introduction, citations

# Execute pipeline
if __name__ == "__main__":
    intro, refs = run_complete_pipeline()

In [ ]:
# Preview outputs
print("\n" + "="*60)
print("INTRODUCTION PREVIEW (First 1500 characters)")
print("="*60)
if 'intro' in locals():
    print(intro[:1500])
    print("\n[... continued in file ...]")
else:
    print("Run the pipeline first to generate introduction")

print("\n" + "="*60)
print("SAMPLE CITATIONS (First 5)")
print("="*60)
if 'refs' in locals():
    for ref in refs[:5]:
        print(ref)
else:
    print("Run the pipeline first to generate citations")